# 01 — Cách dùng module dữ liệu & đánh giá chung (G1-63)

Notebook này cho TV1, TV2 thấy cách lấy cửa sổ, đảo chuẩn hoá, tính độ đo và ghi kết quả **đúng format chung**. Ví dụ dùng một mô hình Ridge rất đơn giản trên ablation (a) để minh hoạ — không phải kết quả chính thức.

Chạy trên Kaggle: clone repo, `%cd` vào gốc repo, rồi chạy từ trên xuống.

In [1]:
import os, sys
# tìm gốc repo (thư mục có configs/data.yaml) dù notebook nằm ở notebooks/bobo/
while not os.path.exists('configs/data.yaml') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd
from src.data.pipeline import prepare
from src.data.windows import feature_columns
from src.data.scaling import inverse_pm25
from src.eval.metrics import wide_to_long, evaluate_grouped, add_area_type
from src.eval.results import results_rows, make_run_id
import warnings; warnings.filterwarnings('ignore')

## 1. Chuẩn bị dữ liệu cho một thí nghiệm
`prepare` đọc lược đồ chung → lấy mốc chia từ `configs/data.yaml` → **fit scaler chỉ trên train** → thêm cột `pm25_z` và `<biến>_s`.

In [2]:
P = prepare('hcm')            # fold=1..4 cho rolling-origin; train_frac=0.5; neighbors=True cho ablation d
print(P.rng)
P.df[['station_id','time','pm25','pm25_z','is_imputed']].dropna().head()

{'train': (Timestamp('2021-02-23 00:00:00'), Timestamp('2021-12-31 23:00:00')), 'val': (Timestamp('2022-01-01 00:00:00'), Timestamp('2022-02-28 23:00:00')), 'test': (Timestamp('2022-03-01 00:00:00'), Timestamp('2022-06-21 23:00:00'))}


,station_id,time,pm25,pm25_z,is_imputed
0,hcm_01,2021-02-23 21:00:00,15.604762,-0.497909,0
1,hcm_01,2021-02-23 22:00:00,14.594118,-0.637328,0
2,hcm_01,2021-02-23 23:00:00,13.436667,-0.808554,0
3,hcm_01,2021-02-24 00:00:00,12.365000,-0.979802,0
4,hcm_01,2021-02-24 01:00:00,11.636667,-1.104214,0


## 2. Tạo cửa sổ trượt
`feature_columns(ablation)` trả về danh sách đặc trưng thống nhất cho (a)–(d).

In [3]:
feats = feature_columns('a')
ws_tr = P.windows('train', feats, L=24)
ws_te = P.windows('test',  feats, L=24)
print(len(ws_tr), len(ws_te), ws_tr.feature_cols)
x, y, m = ws_tr[0]; print(x.shape, y.shape, m.shape)
ws_te.meta().head()

26910 10326 ['pm25_z', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
(24, 5) (24,) (24,)


,city,station_id,time_origin
0,hcm,hcm_01,2022-02-28 23:00:00
1,hcm,hcm_01,2022-03-01 00:00:00
2,hcm,hcm_01,2022-03-01 01:00:00
3,hcm,hcm_01,2022-03-01 02:00:00
4,hcm,hcm_01,2022-03-01 03:00:00


## 3. Một mô hình minh hoạ (Ridge trên input làm phẳng), dự báo cả 24 tầm

In [4]:
from sklearn.linear_model import Ridge
Xtr, Ytr, Mtr = ws_tr.materialize()
keep = Mtr.all(axis=1)                         # Ridge cần đủ 24 nhãn
model = Ridge(alpha=1.0).fit(Xtr[keep].reshape(keep.sum(), -1), Ytr[keep])
Xte, _, _ = ws_te.materialize()
z_pred = model.predict(Xte.reshape(len(Xte), -1))
y_pred = inverse_pm25(z_pred, ws_te.meta()['station_id'], P.params, 'hcm')   # -> µg/m³
y_true = ws_te.y_true_ug()                      # NaN ở giờ không phải đo thật
y_pred.shape, np.isnan(y_true).mean().round(3)

((10326, 24), np.float64(0.035))

## 4. Đánh giá bằng hàm chung + ghi kết quả đúng format

In [5]:
run_id = make_run_id(model='Ridge_demo', ablation='a', L=24)
preds = wide_to_long(ws_te.meta(), y_true, y_pred, horizons=(1, 6, 24), model='Ridge_demo', config=run_id)
display(evaluate_grouped(preds, ['horizon']).round(3))
display(evaluate_grouped(add_area_type(preds, P.cfg), ['horizon', 'area_type']).round(2).head(6))

,horizon,n,MAE,RMSE,R2,MAE_high,n_high,AQI_acc,AQI_macroF1
0,1,10269,3.014,5.170,0.799,9.359,861,0.887,0.556
1,6,10123,6.167,9.416,0.335,21.793,845,0.781,0.267
2,24,9674,6.860,10.383,0.217,24.433,815,0.767,0.230


,horizon,area_type,n,MAE,RMSE,R2,MAE_high,n_high,AQI_acc,AQI_macroF1
0,1,industrial,1225,4.21,7.48,0.71,12.22,133,0.82,0.51
1,1,residential,1930,4.05,6.51,0.77,8.66,366,0.88,0.61
2,1,traffic,3229,2.10,3.41,0.82,8.91,92,0.90,0.56
3,1,traffic_residential,1733,2.36,3.81,0.79,7.39,100,0.92,0.53
4,1,urban_background,2152,3.30,5.34,0.80,10.02,170,0.88,0.68
5,6,industrial,1195,8.28,12.90,0.15,28.00,129,0.67,0.23


In [6]:
rows = results_rows(preds, member='Bo bo', model='Ridge_demo', strategy='S0', ablation='a', train_frac=1.0,
                   L=24, seed=0, eval_set='test', run_id=run_id, notes='demo notebook')
rows.head()   # append_results(rows, 'results/<tên-phương-pháp>.csv') để ghi thật; save_predictions(...) cho file dự đoán

,member,model,strategy,ablation,train_frac,horizon,station,L,seed,MAE,RMSE,R2,MAE_high,AQI_acc,AQI_macroF1,notes,eval_set,n,run_id
0,Bo bo,Ridge_demo,S0,a,1.0,1,ALL,24,0,3.014185,5.169531,0.798888,9.358846,0.886941,0.555827,demo notebook,test,10269,Ridge_demo_2983a956
1,Bo bo,Ridge_demo,S0,a,1.0,6,ALL,24,0,6.166958,9.416399,0.334760,21.792537,0.780599,0.267432,demo notebook,test,10123,Ridge_demo_2983a956
2,Bo bo,Ridge_demo,S0,a,1.0,24,ALL,24,0,6.860487,10.382948,0.217456,24.432935,0.767211,0.229568,demo notebook,test,9674,Ridge_demo_2983a956
3,Bo bo,Ridge_demo,S0,a,1.0,1,hcm_01,24,0,3.298519,5.344798,0.795647,10.017308,0.884758,0.682197,demo notebook,test,2152,Ridge_demo_2983a956
4,Bo bo,Ridge_demo,S0,a,1.0,1,hcm_02,24,0,2.476636,4.042461,0.748007,9.945581,0.911111,0.516857,demo notebook,test,1620,Ridge_demo_2983a956


## 5. So với baseline persistence (bắt buộc)

In [7]:
res = pd.read_csv('results/baselines.csv')
res[(res.station=='ALL') & (res.eval_set=='test') & res.model.isin(['persistence','hourly_mean'])][['model','horizon','n','MAE','RMSE','R2']]

,model,horizon,n,MAE,RMSE,R2
0,persistence,1,10269,3.1345,5.4857,0.7735
1,persistence,6,10123,7.9278,12.0313,-0.0860
2,persistence,24,9674,8.0577,12.6580,-0.1630
21,hourly_mean,1,10269,8.2854,10.8420,0.1154
22,hourly_mean,6,10123,8.2874,10.8500,0.1168
23,hourly_mean,24,9674,8.3267,10.9787,0.1251
